> 在线讲义：[第 3 章 · 折扣 discount γ：远处的奖励打个折](https://wikieden.github.io/rl-from-scratch/lessons/03-discount.html) · 本书《从零学强化学习》

# 第 3 章 · 折扣 discount γ：远处的奖励打个折

本章回答：没完没了的任务，回报会加到无穷大，怎么办？为什么"明天的 1 块钱不如今天的 1 块钱"？回报能不能倒着算？

配套讲义：仓库里 `cs285-deep-rl/notes/lessons/03-discount.md`（讲解、动画、练习都在讲义里）。

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU。

In [ ]:
print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 代码 | 跑出来 |
|---|---|---|
| 一项项加 | `discounted_return([1, 0, 1], 0.9)` | 1.81 |
| 倒着推 | `returns_backward([1, 0, 1], 0.9)` | `[1.81, 0.9, 1.0, 0.0]` |
| γ = 0 / γ = 1 | `discounted_return(r, 0.0)` / `(r, 1.0)` | 1.0 / 2.0 |
| 每步 +1 加 1000 项 | `discounted_return([1] * 1000, 0.9)` | 10.0（= 1/(1−0.9)） |

## 1. 先把要用的东西定义好

这几格只是定义（环境、函数），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**函数 `discounted_return`**（来自 `ch03_discount.py`）：按定义一项一项加：第 k 个奖励乘 γ^k。

In [ ]:
def discounted_return(rewards, gamma):
    """按定义一项一项加：第 k 个奖励乘 γ^k。"""
    return sum(gamma**k * r for k, r in enumerate(rewards))

**函数 `returns_backward`**（来自 `ch03_discount.py`）：从最后往前推：G_T = 0，G_t = R_{t+1} + γ G_{t+1}。返回 [G_0, ..., G_T]。

In [ ]:
def returns_backward(rewards, gamma):
    """从最后往前推：G_T = 0，G_t = R_{t+1} + γ G_{t+1}。返回 [G_0, ..., G_T]。"""
    G = [0.0] * (len(rewards) + 1)
    for t in reversed(range(len(rewards))):
        G[t] = rewards[t] + gamma * G[t + 1]
    return G

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
r = [1, 0, 1]

In [ ]:
print("γ=0.9，奖励 1, 0, 1：G0 =", round(discounted_return(r, 0.9), 4))

In [ ]:
print("倒着算 G0..G3 =", [round(g, 4) for g in returns_backward(r, 0.9)])

In [ ]:
print("γ=0（近视）G0 =", discounted_return(r, 0.0), "；γ=1（不打折）G0 =", discounted_return(r, 1.0))

In [ ]:
long_sum = discounted_return([1] * 1000, 0.9)

In [ ]:
print("每步都 +1、γ=0.9，加 1000 项 =", round(long_sum, 6), "；公式 1/(1−γ) =", round(1 / (1 - 0.9), 6))

## 3. 代码里才看得到的

倒着推的循环在后面会反复出现：CS 285 HW2 的 reward-to-go（第 21 章）、GAE（第 22 章）都是同一个 `reversed` 循环。

## 4. 自己改一改

讲义里建议的几个小实验（数字是实跑出来的，改完对一对）：

1. 奖励只在第 4 步给：`returns_backward([0, 0, 0, 1], 0.9)` → `[0.729, 0.81, 0.9, 1.0, 0.0]`；把 γ 换成 0.5 → `[0.125, 0.25, 0.5, 1.0, 0.0]`。γ 越小，远处的奖励传回来时缩得越厉害。
2. `discounted_return([1] * 2000, 0.99)` → 100.0，正好是 $1/(1-0.99)$。γ 从 0.9 调到 0.99，"看得见"的步数从大约 10 步变成大约 100 步。

讲义末尾的练习题也可以在下面的空格子里用代码验算：先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> 往后一步打一折，第 k 步乘 γ 的 k 次方；
> γ 为零最近视，γ 为一不打折；
> 每步加一总数是 1/(1−γ)；
> 今天总价 = 明天到手 + γ × 明天往后，倒着推最省事。